<a href="https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



My chosen lane is **Refresh / Content Opportunity Scoring**.

This is primarily a **ranking and scoring** task. The goal is not only to classify a page as declining or non-declining. The goal is to rank pages by which ones should be reviewed first by an editor. Each page will receive a priority score based on observable search and content signals. The final output will be a ranked review queue with reason codes, so that a human can understand why a page was prioritized.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



For the starter-data framing exercise, I will use a declining-page proxy:

`is_declining_proxy = 1` when `trend_direction` is `down`, otherwise `0`.

This proxy helps demonstrate how a score can prioritize pages that show a decline pattern in the starter dataset. However, `trend_direction` and `trend_pct` will not be model features because they are directly related to the proxy label.

In the later warehouse-data phase, I will replace this starter proxy with a more honest future-window outcome. The model features will come from an earlier period, while the target will measure whether the page shows a sustained decline or opportunity in a later period.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

The primary success metric will be **Precision@K**.

Precision@K measures how many of the top K pages in the ranked review queue match the declining-page proxy or later future-window outcome. This metric fits the real workflow because an editor has limited time and reviews only the highest-priority pages first.

I will report Precision@20 and Precision@50. Precision@20 represents a smaller review capacity, while Precision@50 represents a larger review queue. The model or scoring method should be compared with the same transparent baseline rule on the same evaluation split.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



The unit of analysis is **one pseudonymized content page**.

Each row represents one content item and its trailing 90-day search and content measurements. The row can include signals such as impressions, average position, CTR, content age, time since the last update, word count, and content type. For this starter-data exercise, I add `is_declining_proxy` only to show what the target column looks like.

In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd

repo_name = "flyrank-ml-internship-starter"
data_file = Path("data/raw/content_refresh_anonymized.csv")

if not data_file.exists():
    if not Path(repo_name).exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
             repo_name],
            check=True
        )
    os.chdir(repo_name)
    data_file = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_file)

# Starter-only proxy target: do not use trend_direction or trend_pct as model features.
df["is_declining_proxy"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

unit_of_analysis = df[
    [
        "content_id",
        "content_type",
        "content_age_days",
        "impressions_90d",
        "avg_position",
        "ctr",
        "days_since_last_update",
        "is_declining_proxy",
    ]
].head(10)

print("One row = one pseudonymized content page with trailing-90-day measurements.")
display(unit_of_analysis)

One row = one pseudonymized content page with trailing-90-day measurements.


,content_id,content_type,content_age_days,impressions_90d,avg_position,ctr,days_since_last_update,is_declining_proxy
0,content_304f48230142,keyword article,187,3803,10.6,0.76,20,1
1,content_a1fb4e703a9e,keyword article,445,15320,20.3,0.05,25,1
2,content_9aa793d4d895,keyword article,141,12581,36.5,0.09,20,1
3,content_331d6c4de07b,keyword article,463,11751,6.2,0.49,22,0
4,content_d99b7a2d90ca,keyword article,263,19140,44.0,0.13,14,1
5,content_d4084a4bc775,keyword article,147,3970,8.5,0.03,20,1
6,content_9a34b442b552,keyword article,90,20,7.0,0.00,20,1
7,content_a63219c6e95a,keyword article,445,1724,21.2,0.06,22,0
8,content_5e6c160719bc,keyword article,90,32574,46.0,0.09,20,1
9,content_c27558df2b0c,keyword article,257,1240,4.9,0.16,104,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can be useful as a transparent baseline, such as prioritizing pages that are both stale and visible. However, a single rule cannot easily represent the interaction between multiple signals.

For example, a page may have high impressions but a weak CTR, a good position but falling visibility, or an old update date but stable performance. These combinations can have different meanings. A scoring model can combine several observable signals and rank pages more consistently than one threshold alone.

The model must still be compared with the fixed rule on the same evaluation split. If the model does not improve the ranking, the simpler rule remains the better choice.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.